# Benchmarking Ensemble Learning Algorithms — Google Colab Edition

Run this notebook top to bottom in Colab. It will:
1. Clone your GitHub repo (or upload files directly — see Cell 2 options)
2. Install dependencies
3. Run the full benchmark pipeline (data → models → metrics → plots)
4. Display all 4 required visualizations inline

**Tip:** In Colab, go to *Runtime → Change runtime type* and confirm you're on a
standard CPU runtime (no GPU needed for this project) with high-RAM if available
(*Runtime → Change runtime type → High-RAM* if your account has that option).


## 1. Get the project files

**Option A — clone from GitHub** (recommended, run this cell):

In [ ]:
# Replace with your repo URL after you've pushed the project to GitHub
REPO_URL = "https://github.com/<your-username>/ensemble-benchmark.git"

import os
if not os.path.exists("ensemble-benchmark"):
    !git clone {REPO_URL}
%cd ensemble-benchmark
!ls

**Option B — no GitHub yet? Upload the zip instead.**
Skip the cell above, run this one, and select `ensemble-benchmark.zip` when prompted.

In [ ]:
from google.colab import files
import zipfile, os

# Uncomment to use this path instead of git clone:
# uploaded = files.upload()
# zip_name = list(uploaded.keys())[0]
# with zipfile.ZipFile(zip_name, 'r') as z:
#     z.extractall('.')
# %cd ensemble-benchmark
# !ls

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt
print("Dependencies installed.")

## 3. Set up imports & speed mode

Colab runs cells from the repo root, so we add `src/` to the path (not `../src`).

**`FAST_MODE`** controls how big the models are. `True` (default) caps every model at 60 trees/rounds and shrinks the scalability sweep — the whole notebook finishes in roughly 5-10 minutes on Colab's free CPU. Set it to `False` once you've confirmed everything runs, if you want the full-size run for your actual report numbers (expect 20-40+ minutes, still much faster than a slow local machine).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("src"))

import warnings
warnings.filterwarnings("ignore")

FAST_MODE = True  # True = quick demo run (~5-10 min). False = full-size run (~20-40+ min).

from data_pipeline import load_raw_data, make_splits
from models import get_baseline_models as _get_baseline_models
from benchmark import run_full_benchmark, run_scalability_sweep, get_learning_curve, get_feature_importances
from tuning import tune_all_models
from plots import (
    plot_speed_vs_performance,
    plot_scalability_curves,
    plot_loss_convergence,
    plot_feature_importance_divergence,
)

def get_baseline_models():
    """Wraps the real model factory; caps n_estimators when FAST_MODE is on."""
    models = _get_baseline_models()
    if FAST_MODE:
        for m in models.values():
            if hasattr(m, "n_estimators"):
                m.set_params(n_estimators=60)
    return models

print(f"Imports OK. FAST_MODE={FAST_MODE}")

## 4. Load & preprocess data

Downloads the Covertype dataset (581,012 rows x 54 features) and caches it under `data/`. Colab's network is generally faster/more reliable than a home connection for this.

In [ ]:
df = load_raw_data()
print(f"Shape: {df.shape}")
df["Cover_Type"].value_counts(normalize=True).sort_index()

In [ ]:
splits = make_splits(df)
X_train, y_train = splits["X_train"], splits["y_train"]
X_val, y_val = splits["X_val"], splits["y_val"]
X_test, y_test = splits["X_test"], splits["y_test"]

print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

## 5. Baseline benchmark (full training set)

Trains all four models and records timing, memory, and accuracy/F1. On Colab's CPU runtime this typically takes a few minutes for the full ~400k-row training set.

In [ ]:
baseline_models = get_baseline_models()
results_df = run_full_benchmark(baseline_models, X_train, y_train, X_test, y_test)
results_df

## 6. (Optional) Hyperparameter tuning

Slow — several minutes to tens of minutes depending on `n_iter`. Skip if you're short on time.

In [ ]:
RUN_TUNING = False  # set True to tune (slow)

if RUN_TUNING:
    tuned_models, best_params = tune_all_models(X_train, y_train, n_iter=20, cv=3)
    models_for_analysis = tuned_models
else:
    models_for_analysis = get_baseline_models()

## 7. Deliverable 1 — Speed vs. Performance Frontier

In [ ]:
plot_speed_vs_performance(results_df)

## 8. Deliverable 2 — Scalability Curves

Retrains fresh models at increasing row counts to trace how training time scales. Under `FAST_MODE`, this uses smaller row sizes (2k/8k/20k/50k) and finishes in 1-2 minutes. Set `FAST_MODE = False` above (and re-run from the imports cell) for the full 10k/50k/100k/500k sweep used in the actual proposal.

In [ ]:
sweep_sizes = (2_000, 8_000, 20_000, 50_000) if FAST_MODE else (10_000, 50_000, 100_000, 500_000)

sweep_df = run_scalability_sweep(
    get_baseline_models, X_train, y_train, X_test, y_test,
    row_sizes=sweep_sizes,
)
sweep_df

In [ ]:
plot_scalability_curves(sweep_df)

## 9. Deliverable 3 — Loss Convergence Profiles

In [ ]:
curves = {}
rf_stages = [5, 15, 30, 45, 60] if FAST_MODE else None  # None = default [10,25,50,75,100,150,200]

for name, model in get_baseline_models().items():
    try:
        stages = rf_stages if name == "RandomForest" else None
        train_loss, val_loss = get_learning_curve(name, model, X_train, y_train, X_val, y_val, max_stages=stages)
        if train_loss and val_loss:
            curves[name] = (train_loss, val_loss)
    except MemoryError:
        print(f"Skipped learning curve for {name}: ran out of memory.")

plot_loss_convergence(curves)

## 10. Deliverable 4 — Feature Importance Divergence

In [ ]:
importances = {}
for name, model in models_for_analysis.items():
    fitted = model.fit(X_train, y_train)
    importances[name] = get_feature_importances(name, fitted, X_train.columns)

plot_feature_importance_divergence(importances)

## 11. Download your results

Zips up `results/` (figures + metrics CSVs) so you can download them to your machine or attach them to your project submission.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("results", "zip", "results")
files.download("results.zip")

## 12. Findings (fill in after running)

- **Computational efficiency:** ...
- **Predictive performance:** ...
- **Overfitting & generalization:** ...
- **Feature interpretation:** ...
